# Week 4: LLM Benchmarking vs ML Model

**Team:** Zero Shot Hopefuls  
**Task:** Predict 1-week-ahead compound heatwave+drought events (`target_compound_1wk`, binary 0/1) from climate observations.

This notebook benchmarks three LLM endpoints on the same prediction task and test split used in Week 3's ML models (LightGBM and Logistic Regression). The challenge is that our data is **tabular/numerical** (26 climate features), not natural language — so prompts must carefully present feature vectors with domain context.

**LLM Endpoints (JupyterHub):**
- Phi-3.5-mini-instruct (port 8000)
- Phi-mini-MoE-instruct (port 8001)
- gemma-3-12b-it (port 8002)

**Week 3 ML Baseline (on full test set):**
- LightGBM: F1=0.6432, AUC=0.9886
- Logistic Regression: F1=0.5790, AUC=0.9813

# AI-GENERATED: This notebook was generated by Claude (Anthropic) and reviewed
# by the Zero Shot Hopefuls team.

In [ ]:
# Install required packages (run once on JupyterHub if not already installed)
!pip install pandas openai scikit-learn numpy
%pip install scikit-learn

## Step 1: Imports and Configuration

Sets up all libraries, model endpoints, file paths, and feature definitions in one place.
Changing a path or adding a model only requires editing this cell.

In [ ]:
# AI-GENERATED: Claude (Anthropic). Setup cell for LLM benchmarking pipeline.
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd

openai = __import__('openai')
OpenAI = openai.OpenAI

from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    brier_score_loss, confusion_matrix, classification_report,
)

# ---------------------------------------------------------------------------
# Paths — adjust TEAM_DIR if running from a different location
# ---------------------------------------------------------------------------
TEAM_DIR = Path('..').resolve()  # zero-shot-hopefuls/
DATA_DIR = TEAM_DIR / 'Data'
MODELS_DIR = TEAM_DIR / 'Models'
PROMPTS_DIR = TEAM_DIR / 'Prompts'
RESULTS_DIR = TEAM_DIR / 'Results'

SUBSET_CSV = DATA_DIR / 'llm_test_subset.csv'
FEW_SHOT_JSON = PROMPTS_DIR / 'few_shot_samples.json'

# ---------------------------------------------------------------------------
# Model endpoints (PNNL Research Computing JupyterHub)
# ---------------------------------------------------------------------------
MODEL_ENDPOINTS = [
    {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it',        'model_name': 'gemma-3-12b-it',        'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

# ---------------------------------------------------------------------------
# Feature definitions (same 26 features used in Week 3 ML models)
# ---------------------------------------------------------------------------
TARGET_COL = 'target_compound_1wk'

FEATURE_COLS = [
    'tmax', 'tmin', 'tmean', 'HI02', 'HI05',
    'FHS_c9', 'maxFRP_c9', 'BA_km2',
    'pdsi', 'pdsi_category', 'spi90d', 'spei90d',
    'air_sfc', 'apcp', 'soilm', 'lhtfl', 'shtfl', 'rhum_2m',
    'BCOC', 'LAI', 'NDVI', 'FWI',
    'wind_speed', 'seasonal_sin', 'seasonal_cos',
    'tmean_7d_avg', 'drought_trend_14d', 'fwi_7d_max',
]

# Top features from Week 3 SHAP analysis (for simplified prompt)
TOP_FEATURES = ['pdsi', 'seasonal_cos', 'tmax', 'HI02', 'BCOC', 'FWI', 'spi90d', 'spei90d']

# Week 3 optimal thresholds
LR_THRESHOLD = 0.86
LGB_THRESHOLD = 0.65

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Team directory: {TEAM_DIR}')
print(f'Subset CSV: {SUBSET_CSV}')
print(f'Few-shot JSON: {FEW_SHOT_JSON}')

## Step 2: Load Data

Loads the 1,000-row stratified test subset (sampled from Week 3's 2018-2020 test split) and
ML baseline predictions that were pre-computed using the saved Week 3 models.
Also loads the few-shot examples selected from the training set (2001-2015, no leakage).

In [ ]:
# AI-GENERATED: Claude (Anthropic). Data loading for LLM benchmark.
# Load test subset with pre-computed ML predictions
df = pd.read_csv(SUBSET_CSV)

print(f'Loaded {len(df)} rows')
print(f'Positive rate: {df[TARGET_COL].sum()} / {len(df)} = {100 * df[TARGET_COL].mean():.2f}%')
print(f'Years: {sorted(df["year"].unique())}')

# Load few-shot examples
with open(FEW_SHOT_JSON) as f:
    few_shot_examples = json.load(f)
print(f'\nFew-shot examples: {len(few_shot_examples)} '
      f'({sum(e["label"] for e in few_shot_examples)} positive, '
      f'{sum(1 - e["label"] for e in few_shot_examples)} negative)')

# Verify ML baselines are present
print(f'\nML baseline columns present:')
for col in ['ml_lr_prediction', 'ml_lr_probability', 'ml_lgb_prediction', 'ml_lgb_probability']:
    print(f'  {col}: {col in df.columns}')

display(df[['lat', 'lon', 'year', 'week', 'tmax', 'pdsi', 'HI02', TARGET_COL,
            'ml_lgb_prediction', 'ml_lgb_probability']].head(10))

## Step 3: Prompt Builder Functions

Three prompt versions to test:
- **v1 (zero-shot):** All 26 features with descriptions and units. No examples.
- **v2 (few-shot):** Same as v1, but prepends labeled examples from training data.
- **v3 (simplified):** Only the top 8 SHAP-important features. Less noise for small LLMs.

All versions constrain output to strict JSON: `{"prediction": 0 or 1, "confidence": 0.0-1.0}`

In [ ]:
# AI-GENERATED: Claude (Anthropic). Prompt builder functions for tabular climate data.

SYSTEM_PROMPT = (
    'You are a climate hazard forecasting system that predicts compound '
    'heatwave+drought events from numerical climate observations. '
    'Always respond with valid JSON only — no extra text.'
)

# Feature descriptions with units for human-readable prompts
FEATURE_INFO = {
    'tmax': ('Max temperature', 'C'),
    'tmin': ('Min temperature', 'C'),
    'tmean': ('Mean temperature', 'C'),
    'HI02': ('Heatwave active (tmean-based)', '0/1'),
    'HI05': ('Heatwave active (tmax-based)', '0/1'),
    'FHS_c9': ('Fire hotspot count', 'pixels'),
    'maxFRP_c9': ('Max fire radiative power', 'MW'),
    'BA_km2': ('Burned area', 'km2'),
    'pdsi': ('Palmer Drought Severity Index', '-5=extreme drought to +5=extreme wet'),
    'pdsi_category': ('PDSI drought category', '0=extreme drought, 5=normal, 10=extreme wet'),
    'spi90d': ('90-day Standardized Precipitation Index', 'negative=dry'),
    'spei90d': ('90-day Standardized Precip-Evapotranspiration Index', 'negative=dry'),
    'air_sfc': ('Surface air temperature', 'K'),
    'apcp': ('Precipitation', 'kg/m2'),
    'soilm': ('Soil moisture', 'kg/m2'),
    'lhtfl': ('Latent heat flux', 'W/m2'),
    'shtfl': ('Sensible heat flux', 'W/m2'),
    'rhum_2m': ('Relative humidity at 2m', '%'),
    'BCOC': ('Biomass burning aerosol concentration', ''),
    'LAI': ('Leaf Area Index', 'm2/m2'),
    'NDVI': ('Vegetation Index', '-1 to 1'),
    'FWI': ('Fire Weather Index', 'higher=more risk'),
    'wind_speed': ('Wind speed', 'm/s'),
    'seasonal_sin': ('Seasonal cycle (sin)', ''),
    'seasonal_cos': ('Seasonal cycle (cos)', ''),
    'tmean_7d_avg': ('7-day rolling mean temperature', 'C'),
    'drought_trend_14d': ('14-day PDSI change', 'negative=worsening drought'),
    'fwi_7d_max': ('7-day max Fire Weather Index', ''),
}


def _format_features(row, feature_list):
    """Format a row's features as readable text lines."""
    lines = []
    for feat in feature_list:
        desc, unit = FEATURE_INFO[feat]
        val = row[feat]
        unit_str = f' ({unit})' if unit else ''
        lines.append(f'- {desc}{unit_str}: {val:.4f}')
    return '\n'.join(lines)


def _format_example(example, feature_list):
    """Format a single few-shot example."""
    lines = []
    for feat in feature_list:
        desc, unit = FEATURE_INFO[feat]
        val = example['features'][feat]
        unit_str = f' ({unit})' if unit else ''
        lines.append(f'- {desc}{unit_str}: {val:.4f}')
    label = example['label']
    return '\n'.join(lines) + f'\n\nAnswer: {{"prediction": {label}, "confidence": 0.95}}'


def make_prompt(row, version='v1', few_shot=None):
    """Build a prompt for a single data row.

    Parameters
    ----------
    row : pd.Series — one row from the test subset
    version : str — 'v1' (zero-shot, all features), 'v2' (few-shot, all features),
                     'v3' (zero-shot, top features), 'v3_few' (few-shot, top features)
    few_shot : list[dict] — few-shot examples (only used for v2/v3_few)
    """
    use_top_only = version.startswith('v3')
    use_few_shot = version in ('v2', 'v3_few')
    features = TOP_FEATURES if use_top_only else FEATURE_COLS

    context = (
        'Predict whether a compound heatwave+drought event will occur NEXT WEEK.\n'
        'A compound event = heatwave (tmean > 95th percentile for 3+ days) '
        'AND drought (PDSI category <= 3).\n'
        'Respond with ONLY JSON: {"prediction": 0 or 1, "confidence": <0.0-1.0>}\n'
    )

    parts = [context]

    # Few-shot examples
    if use_few_shot and few_shot:
        parts.append('--- EXAMPLES ---')
        for i, ex in enumerate(few_shot, 1):
            parts.append(f'\nExample {i}:')
            parts.append(_format_example(ex, features))
        parts.append('\n--- NOW PREDICT ---\n')

    # Current observation
    parts.append('Current week observations:')
    parts.append(_format_features(row, features))
    parts.append('\nAnswer:')

    return '\n'.join(parts)


# Quick test: show what v1 and v2 prompts look like
sample_row = df.iloc[0]
print('=== v1 (zero-shot, all features) — first 500 chars ===')
print(make_prompt(sample_row, version='v1')[:500])
print(f'\n... ({len(make_prompt(sample_row, version="v1"))} total chars)')

print('\n=== v3 (zero-shot, top features) — full ===')
print(make_prompt(sample_row, version='v3'))

## Step 4: Response Parser

Extracts `prediction` (0 or 1) and `confidence` (float) from LLM JSON responses.
Handles common failure modes: invalid JSON, missing keys, out-of-range values, extra text wrapping.

In [ ]:
# AI-GENERATED: Claude (Anthropic). Response parser for LLM JSON outputs.

def parse_response(raw_text):
    """Parse an LLM response into a structured prediction.

    Returns dict with keys: llm_prediction, llm_confidence, parse_ok, parse_error
    """
    output = {
        'llm_prediction': None,
        'llm_confidence': np.nan,
        'parse_ok': False,
        'parse_error': None,
    }

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    # Extract JSON object from response (LLMs sometimes wrap in markdown or extra text)
    m = re.search(r'\{[^{}]*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except (json.JSONDecodeError, ValueError):
        output['parse_error'] = 'invalid_json'
        return output

    # Extract prediction
    pred = payload.get('prediction')
    if pred is None:
        output['parse_error'] = 'missing_prediction_key'
        return output

    try:
        pred = int(pred)
    except (ValueError, TypeError):
        output['parse_error'] = 'non_integer_prediction'
        return output

    if pred not in (0, 1):
        output['parse_error'] = f'invalid_prediction_value_{pred}'
        return output

    # Extract confidence
    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except (ValueError, TypeError):
        conf = np.nan

    if not np.isnan(conf) and not (0.0 <= conf <= 1.0):
        conf = np.nan  # out of range, treat as missing

    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output


# Quick parser tests
test_cases = [
    '{"prediction": 1, "confidence": 0.85}',           # valid
    '{"prediction": 0, "confidence": 0.12}',            # valid
    '```json\n{"prediction": 1, "confidence": 0.9}\n```',  # markdown-wrapped
    'The prediction is {"prediction": 0, "confidence": 0.3} based on...',  # extra text
    '{"prediction": 2, "confidence": 0.5}',             # invalid pred value
    'I think the answer is 0',                           # no JSON
    '',                                                  # empty
]
for tc in test_cases:
    result = parse_response(tc)
    status = 'OK' if result['parse_ok'] else f'FAIL ({result["parse_error"]})'
    print(f'{status:30s} | pred={result["llm_prediction"]} conf={result["llm_confidence"]:.2f}' 
          if result['parse_ok'] else f'{status:30s} | raw: {tc[:50]}')

## Step 5: LLM Query Function and Smoke Test

Tests connectivity to each endpoint with a single example before running the full benchmark.
Uses `temperature=0.0` and `seed=0` for reproducibility.

In [ ]:
# AI-GENERATED: Claude (Anthropic). LLM query function and smoke test.

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    """Send a prompt to an LLM endpoint and return the raw response text."""
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed,
    )
    return response.choices[0].message.content


# Smoke test: query each endpoint with one example
example_row = df.iloc[0]
test_prompt = make_prompt(example_row, version='v1')

print('Smoke test — querying each endpoint with v1 prompt...\n')
for endpoint_cfg in MODEL_ENDPOINTS:
    try:
        raw = query_llm(test_prompt, endpoint_cfg)
        parsed = parse_response(raw)
        status = 'PASS' if parsed['parse_ok'] else f'PARSE FAIL: {parsed["parse_error"]}'
        print(f"Model: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
        print(f"  Status: {status}")
        print(f"  Raw: {raw[:200]}")
        print(f"  Parsed: pred={parsed['llm_prediction']}, conf={parsed['llm_confidence']}")
        print()
    except Exception as e:
        print(f"Model: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
        print(f"  ERROR: {type(e).__name__}: {e}")
        print(f"  (This is expected if not running on JupyterHub with active endpoints)")
        print()

## Step 6: Prompt Iteration on Small Subset

Tests each prompt version (v1, v2, v3, v3_few) on a small subset (~20 rows) across all 3 models.
Compares parse reliability and accuracy to select the best prompt version before the full run.

**WARNING:** This cell queries LLMs — skip if endpoints are not available.

In [ ]:
# AI-GENERATED: Claude (Anthropic). Prompt iteration across versions and models.

# Use a small balanced subset for iteration: sample up to 10 positive + 10 negative
pos_idx = df[df[TARGET_COL] == 1].index.tolist()
neg_idx = df[df[TARGET_COL] == 0].sample(n=min(10, len(df[df[TARGET_COL] == 0])),
                                          random_state=42).index.tolist()
iter_idx = pos_idx[:10] + neg_idx[:10]
iter_subset = df.loc[iter_idx].reset_index(drop=True)
print(f'Iteration subset: {len(iter_subset)} rows '
      f'({iter_subset[TARGET_COL].sum()} positive, '
      f'{(iter_subset[TARGET_COL] == 0).sum()} negative)')

versions = ['v1', 'v2', 'v3', 'v3_few']
iteration_records = []

for endpoint_cfg in MODEL_ENDPOINTS:
    for version in versions:
        print(f"\n{endpoint_cfg['label']} / {version}:")
        for i, (_, row) in enumerate(iter_subset.iterrows()):
            prompt = make_prompt(
                row, version=version,
                few_shot=few_shot_examples if version in ('v2', 'v3_few') else None
            )
            try:
                raw = query_llm(prompt, endpoint_cfg)
                parsed = parse_response(raw)
            except Exception as e:
                raw = f'ERROR: {e}'
                parsed = {'llm_prediction': None, 'llm_confidence': np.nan,
                          'parse_ok': False, 'parse_error': 'endpoint_error'}

            iteration_records.append({
                'model_label': endpoint_cfg['label'],
                'version': version,
                'ground_truth': int(row[TARGET_COL]),
                'llm_prediction': parsed['llm_prediction'],
                'llm_confidence': parsed['llm_confidence'],
                'parse_ok': parsed['parse_ok'],
                'parse_error': parsed['parse_error'],
                'raw_response': raw,
            })
        # Progress dot
        print(f'  Completed {len(iter_subset)} rows')

iter_df = pd.DataFrame(iteration_records)
iter_df['is_correct'] = (iter_df['ground_truth'] == iter_df['llm_prediction'])

# Summary per model x version
iter_summary = iter_df.groupby(['model_label', 'version'], as_index=False).agg(
    parse_rate=('parse_ok', 'mean'),
    accuracy=('is_correct', 'mean'),
    rows=('is_correct', 'size'),
).sort_values(['model_label', 'parse_rate', 'accuracy'], ascending=[True, False, False])

print('\n=== Prompt Iteration Summary ===')
display(iter_summary)

# Select best version per model (highest parse rate, then accuracy)
best_versions = (
    iter_summary.sort_values(['parse_rate', 'accuracy'], ascending=False)
    .groupby('model_label', as_index=False)
    .first()[['model_label', 'version']]
)
print('\nBest prompt version per model:')
display(best_versions)

# Store as dict for use in full benchmark
BEST_VERSION = dict(zip(best_versions['model_label'], best_versions['version']))

## Step 7: Full Benchmark Run

Runs the best prompt version for each model against all 1,000 test rows.
Saves raw responses and parsed predictions. Progress is logged every 50 rows.

**WARNING:** This cell takes a while to run (1,000 rows x 3 models). Run one model at a time if timeouts occur.

In [ ]:
# AI-GENERATED: Claude (Anthropic). Full benchmark run across all models and test rows.

# Fallback if prompt iteration was skipped (endpoints not available):
# default all models to v1 (zero-shot, all features)
if 'BEST_VERSION' not in dir() or not BEST_VERSION:
    BEST_VERSION = {cfg['label']: 'v1' for cfg in MODEL_ENDPOINTS}
    print('Using default v1 prompt for all models (prompt iteration was skipped)')

all_records = []

for endpoint_cfg in MODEL_ENDPOINTS:
    model_label = endpoint_cfg['label']
    version = BEST_VERSION.get(model_label, 'v1')
    use_few_shot = version in ('v2', 'v3_few')

    print(f"\n{'='*60}")
    print(f"Running benchmark: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print(f"Prompt version: {version}")
    print(f"{'='*60}")

    start_time = time.time()

    for i, (_, row) in enumerate(df.iterrows()):
        prompt = make_prompt(
            row, version=version,
            few_shot=few_shot_examples if use_few_shot else None
        )
        try:
            raw = query_llm(prompt, endpoint_cfg)
            parsed = parse_response(raw)
        except Exception as e:
            raw = f'ERROR: {e}'
            parsed = {'llm_prediction': None, 'llm_confidence': np.nan,
                      'parse_ok': False, 'parse_error': 'endpoint_error'}

        all_records.append({
            'row_index': i,
            'model_label': model_label,
            'model_name': endpoint_cfg['model_name'],
            'endpoint_port': endpoint_cfg['port'],
            'prompt_version': version,
            'ground_truth': int(row[TARGET_COL]),
            'ml_lgb_prediction': int(row['ml_lgb_prediction']),
            'ml_lgb_probability': float(row['ml_lgb_probability']),
            'ml_lr_prediction': int(row['ml_lr_prediction']),
            'ml_lr_probability': float(row['ml_lr_probability']),
            'raw_response': raw,
            **parsed,
        })

        if (i + 1) % 50 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            eta = (len(df) - i - 1) / rate
            print(f'  [{i+1:4d}/{len(df)}] {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining '
                  f'({rate:.1f} rows/s)')

    elapsed = time.time() - start_time
    print(f'  Completed {len(df)} rows in {elapsed:.0f}s')

results_df = pd.DataFrame(all_records)
print(f'\nTotal results: {len(results_df)} rows ({len(results_df) // len(df)} models)')

# Quick check: parse success rates
parse_summary = results_df.groupby('model_label')['parse_ok'].mean()
print('\nParse success rates:')
for model, rate in parse_summary.items():
    print(f'  {model}: {rate:.1%}')

## Step 8: Compute Evaluation Metrics

Computes the same metrics used in Week 3 (F1, AUC-ROC, Precision, Recall, Brier Score)
for each LLM and the ML baselines on the identical 1,000-row subset.
Displays a side-by-side comparison table.

In [ ]:
# AI-GENERATED: Claude (Anthropic). Evaluation metrics computation and ML vs LLM comparison.

def compute_metrics(y_true, y_pred, y_prob=None, label='model'):
    """Compute classification metrics matching Week 3 evaluate.py."""
    metrics = {
        'model': label,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None and not np.all(np.isnan(y_prob)):
        valid = ~np.isnan(y_prob)
        if valid.sum() > 0 and len(np.unique(y_true[valid])) > 1:
            metrics['auc_roc'] = roc_auc_score(y_true[valid], y_prob[valid])
            metrics['brier_score'] = brier_score_loss(y_true[valid], y_prob[valid])
        else:
            metrics['auc_roc'] = np.nan
            metrics['brier_score'] = np.nan
    else:
        metrics['auc_roc'] = np.nan
        metrics['brier_score'] = np.nan
    return metrics


# --- ML baselines on the same subset ---
y_true = df[TARGET_COL].values

ml_metrics = []
ml_metrics.append(compute_metrics(
    y_true, df['ml_lgb_prediction'].values, df['ml_lgb_probability'].values,
    label='LightGBM (Week 3)'
))
ml_metrics.append(compute_metrics(
    y_true, df['ml_lr_prediction'].values, df['ml_lr_probability'].values,
    label='Logistic Regression (Week 3)'
))

# --- LLM metrics ---
llm_metrics = []
for model_label in results_df['model_label'].unique():
    model_data = results_df[results_df['model_label'] == model_label].copy()

    # Only evaluate parseable rows
    valid_mask = model_data['parse_ok'].values
    n_valid = valid_mask.sum()
    n_total = len(model_data)

    if n_valid == 0:
        llm_metrics.append({
            'model': f'{model_label} (LLM)',
            'f1': 0.0, 'precision': 0.0, 'recall': 0.0,
            'auc_roc': np.nan, 'brier_score': np.nan,
            'parse_rate': 0.0, 'n_valid': 0, 'n_total': n_total,
        })
        continue

    y_true_llm = model_data.loc[valid_mask, 'ground_truth'].values
    y_pred_llm = model_data.loc[valid_mask, 'llm_prediction'].values.astype(int)
    y_conf_llm = model_data.loc[valid_mask, 'llm_confidence'].values

    m = compute_metrics(y_true_llm, y_pred_llm, y_conf_llm, label=f'{model_label} (LLM)')
    m['parse_rate'] = n_valid / n_total
    m['n_valid'] = n_valid
    m['n_total'] = n_total
    llm_metrics.append(m)

# --- Combined comparison table ---
all_metrics = ml_metrics + llm_metrics
comparison_df = pd.DataFrame(all_metrics)

# Reorder columns
col_order = ['model', 'f1', 'auc_roc', 'precision', 'recall', 'brier_score']
extra_cols = [c for c in comparison_df.columns if c not in col_order]
comparison_df = comparison_df[col_order + extra_cols]

print('=== ML vs LLM Comparison (on same 1,000-row subset) ===\n')
display(comparison_df.round(4))

# Confusion matrices for LLMs
print('\n=== LLM Confusion Matrices ===')
for model_label in results_df['model_label'].unique():
    model_data = results_df[results_df['model_label'] == model_label]
    valid = model_data[model_data['parse_ok']]
    if len(valid) > 0:
        y_t = valid['ground_truth'].values
        y_p = valid['llm_prediction'].values.astype(int)
        tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0, 1]).ravel()
        print(f'\n{model_label}: TP={tp}, FP={fp}, TN={tn}, FN={fn}')

## Step 9: Error Analysis

Identifies concrete failure examples:
- **Parse failures:** Invalid JSON, extra text, wrong format
- **Prediction errors:** Which feature patterns does the LLM get wrong?
- **Confidence calibration:** Does LLM confidence correlate with correctness?
- **ML vs LLM disagreement:** Do they fail on the same rows or different ones?

In [ ]:
# AI-GENERATED: Claude (Anthropic). Error analysis for LLM benchmark.

print('=== Error Analysis ===\n')

# --- 1. Parse failures ---
parse_failures = results_df[~results_df['parse_ok']].copy()
print(f'Total parse failures: {len(parse_failures)} / {len(results_df)} '
      f'({100 * len(parse_failures) / len(results_df):.1f}%)')

if len(parse_failures) > 0:
    print('\nParse failure breakdown by model:')
    display(parse_failures.groupby(['model_label', 'parse_error']).size()
            .reset_index(name='count'))

    print('\nSample parse failure responses:')
    for _, row in parse_failures.head(5).iterrows():
        print(f"  [{row['model_label']}] error={row['parse_error']}")
        print(f"    raw: {str(row['raw_response'])[:150]}")
        print()
else:
    print('  No parse failures detected.')

# --- 2. Prediction errors ---
print('\n--- Prediction Errors ---')
valid_results = results_df[results_df['parse_ok']].copy()
valid_results['is_correct'] = (valid_results['ground_truth'] == valid_results['llm_prediction'])

for model_label in valid_results['model_label'].unique():
    model_data = valid_results[valid_results['model_label'] == model_label]
    errors = model_data[~model_data['is_correct']]
    n_fp = ((errors['ground_truth'] == 0) & (errors['llm_prediction'] == 1)).sum()
    n_fn = ((errors['ground_truth'] == 1) & (errors['llm_prediction'] == 0)).sum()
    print(f'\n{model_label}:')
    print(f'  Total errors: {len(errors)} / {len(model_data)} ({100*len(errors)/len(model_data):.1f}%)')
    print(f'  False positives (predicted event, none occurred): {n_fp}')
    print(f'  False negatives (missed real event): {n_fn}')

    # Show characteristics of false negatives (missed compound events)
    if n_fn > 0:
        fn_rows = errors[(errors['ground_truth'] == 1) & (errors['llm_prediction'] == 0)]
        fn_indices = fn_rows['row_index'].values
        fn_features = df.iloc[fn_indices][['pdsi', 'tmax', 'HI02', 'FWI', 'spi90d']].describe()
        print(f'  Feature summary of missed events:')
        display(fn_features.round(2))

# --- 3. Confidence calibration ---
print('\n--- Confidence Calibration ---')
for model_label in valid_results['model_label'].unique():
    model_data = valid_results[valid_results['model_label'] == model_label]
    has_conf = model_data[~model_data['llm_confidence'].isna()]
    if len(has_conf) > 0:
        correct = has_conf[has_conf['is_correct']]
        incorrect = has_conf[~has_conf['is_correct']]
        print(f'\n{model_label}:')
        print(f'  Mean confidence (correct): {correct["llm_confidence"].mean():.3f}')
        print(f'  Mean confidence (incorrect): {incorrect["llm_confidence"].mean():.3f}' 
              if len(incorrect) > 0 else '  No incorrect predictions with confidence')
        print(f'  Confidence range: [{has_conf["llm_confidence"].min():.2f}, '
              f'{has_conf["llm_confidence"].max():.2f}]')

# --- 4. LLM vs ML disagreement ---
print('\n--- LLM vs ML Disagreement ---')
for model_label in valid_results['model_label'].unique():
    model_data = valid_results[valid_results['model_label'] == model_label]
    lgb_agrees = (model_data['llm_prediction'].values == model_data['ml_lgb_prediction'].values)
    print(f'\n{model_label} vs LightGBM:')
    print(f'  Agreement rate: {lgb_agrees.mean():.1%}')
    # Cases where LLM is right and ML is wrong
    llm_right_ml_wrong = (
        (model_data['llm_prediction'].values == model_data['ground_truth'].values) &
        (model_data['ml_lgb_prediction'].values != model_data['ground_truth'].values)
    ).sum()
    ml_right_llm_wrong = (
        (model_data['ml_lgb_prediction'].values == model_data['ground_truth'].values) &
        (model_data['llm_prediction'].values != model_data['ground_truth'].values)
    ).sum()
    print(f'  LLM correct, ML wrong: {llm_right_ml_wrong}')
    print(f'  ML correct, LLM wrong: {ml_right_llm_wrong}')

## Step 10: Save Results and Export

Exports per-model raw and clean CSVs following the competition naming convention:
`{model_name}_results_clean_{iteration}.csv` and `{model_name}_results_raw_{iteration}.csv`

In [ ]:
# AI-GENERATED: Claude (Anthropic). Save results CSVs for submission.

# Clean columns for submission
clean_cols = [
    'row_index', 'model_label', 'model_name', 'endpoint_port', 'prompt_version',
    'ground_truth', 'ml_lgb_prediction', 'ml_lgb_probability',
    'llm_prediction', 'llm_confidence', 'parse_ok', 'parse_error',
]
raw_cols = clean_cols + ['raw_response']

# Per-model CSVs
for model_label in results_df['model_label'].unique():
    model_data = results_df[results_df['model_label'] == model_label]

    clean_path = RESULTS_DIR / f'{model_label}_results_clean_{ITERATION_NUMBER}.csv'
    raw_path = RESULTS_DIR / f'{model_label}_results_raw_{ITERATION_NUMBER}.csv'

    model_data[clean_cols].to_csv(clean_path, index=False)
    model_data[raw_cols].to_csv(raw_path, index=False)

    print(f'Saved: {clean_path.name} ({len(model_data)} rows)')
    print(f'Saved: {raw_path.name} ({len(model_data)} rows)')

# Combined results
combined_clean = RESULTS_DIR / f'llm_results_clean_{ITERATION_NUMBER}.csv'
combined_raw = RESULTS_DIR / f'llm_results_raw_{ITERATION_NUMBER}.csv'
results_df[clean_cols].to_csv(combined_clean, index=False)
results_df[raw_cols].to_csv(combined_raw, index=False)
print(f'\nSaved combined: {combined_clean.name}')
print(f'Saved combined: {combined_raw.name}')

# Summary comparison CSV
comparison_path = RESULTS_DIR / f'comparison_summary_{ITERATION_NUMBER}.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f'Saved comparison: {comparison_path.name}')

# --- Draft comparison paragraph ---
print('\n' + '='*60)
print('COMPARISON SUMMARY (draft for README)')
print('='*60)
print()
for _, row in comparison_df.iterrows():
    parts = [f"{row['model']}: F1={row['f1']:.4f}"]
    if not np.isnan(row.get('auc_roc', np.nan)):
        parts.append(f"AUC={row['auc_roc']:.4f}")
    parts.append(f"Precision={row['precision']:.4f}")
    parts.append(f"Recall={row['recall']:.4f}")
    if 'parse_rate' in row and not np.isnan(row.get('parse_rate', np.nan)):
        parts.append(f"Parse={row['parse_rate']:.1%}")
    print(', '.join(parts))

print('\n--- End of benchmark ---')